# 📈 eMFer - Empirical Mutual Fund Evaluation and Research *by Niloy*

"Investing feels smart… that is, until you that realize luck might be doing the heavy lifting."

I came across this quote somewhere, and it stuck. I kept coming back to this thought while looking at my own mutual fund investments.

After a few nights of questioning whether I was actually picking good funds - or just getting lucky - I decided to build something to answer it properly.

Most tools show point-to-point returns. I wanted to go a step further and understand how consistent those returns have actually been over time.

So I built this tool to dig deeper.

---

## What this tool does

- Fetches historical mutual fund NAV data automatically (no manual downloads needed)
- Calculates rolling CAGR across different time periods (as desired by the user)
- Handles missing dates (weekends, holidays, etc.)
- Interactive visualized return distributions instead of just point-to point returns
- Exports structured results for further analysis by an AI Assistant
- Lets you run the tool on different funds and let the AI assistant use stored results for deeper insights and comparisons

---

### Scout - The AI Assistant

Ask Scout anything about your mutual funds.

Analyze a single fund or compare multiple funds - Scout breaks down performance, consistency, and risk in a clear, easy-to-understand way.

---

## To summarise

<p align="center">
  <img src="https://drive.google.com/uc?export=view&id=1VPR0r80Nekf8z74cEyYWaIISfGrWaj62" width="850">
</p>

---

<details>
<summary><strong>FAQs</strong></summary>

### What exactly is rolling CAGR?

It answers:
"What would returns look like if I had invested on different dates in the past?"

So instead of one number, you get a distribution of outcomes.

---


</details>

---

Ready to roll? Let’s go! 🚀🚀🚀

In [ ]:
#@title Clear working directory
# # FULL RESET CELL

# import shutil
# import os

# folders_to_delete = [
#     "project_mf_analyzer"
# ]

# for folder in folders_to_delete:
#     if os.path.exists(folder):
#         shutil.rmtree(folder)

# print("Local folders cleared")

#Stage: User inputs

In [ ]:
#@title Time horizon for Analysis
nav_years = input ('Please enter desired rolling returns window (in years):')
n_years = int(nav_years)

In [ ]:
#@title Select Mutual Fund

import pandas as pd
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

def get_all_schemes():
    url = "https://api.mfapi.in/mf"
    return pd.DataFrame(requests.get(url).json())

schemes = get_all_schemes()

fund_dropdown = widgets.Combobox(
    placeholder="Start typing fund name...",
    options=schemes["schemeName"].sort_values().tolist(),
    description="Fund:",
    ensure_option=True,
    layout=widgets.Layout(width="90%")
)

output = widgets.Output()

# global variable to store selected scheme code
mf_scheme_code = None

def on_fund_select(change):
    global mf_scheme_code

    selected_fund = change["new"]

    selected_row = schemes[
        schemes["schemeName"] == selected_fund
    ].reset_index(drop=True)

    if not selected_row.empty:
        mf_scheme_code = selected_row.loc[0, "schemeCode"]

        with output:
            clear_output()
            print('Fund details: \n')
            display(selected_row)

fund_dropdown.observe(on_fund_select, names="value")

display(fund_dropdown, output)

# ▶️ Run Instructions

    - Click anywhere on this this cell
    - Click '▼' button next to 'Run all' in the top menu
    - Select 'Run focused cell and all cells below'

#Stage: Data fetching and processing

##Stage: Fetching MF NAV data

In [ ]:
#@title Fetching MF NAV data
import requests

def fetch_nav_history(mf_scheme_code):
    url = f'https://api.mfapi.in/mf/{mf_scheme_code}'

    response = requests.get(url)
    data = response.json()

    nav_data = data['data']

    df = pd.DataFrame(nav_data)

    df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
    df['nav'] = pd.to_numeric(df['nav'])

    df = df.sort_values('date')

    fund_name = data['meta']['scheme_name']
    df['fund_name'] = fund_name

    return df, fund_name




In [ ]:
df_nav_all, fund_name = fetch_nav_history(mf_scheme_code)
df_nav_all.head()

In [ ]:
print (fund_name)

## Stage: Creating widgets

In [ ]:
# @title Install ipywidgets
!pip install ipywidgets --quiet
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# @title Define widgets
# Widgets
fund_input = widgets.Text(
    description='Enter name of the Mutual Fund:',
    placeholder='e.g., SBI Small Cap Fund - Regular Plan - Growth',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

nav_path_input = widgets.Text(
    description='Enter the directory where historical NAV data is present: ',
    placeholder='/content/project_mf_analyzer/nav_files',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

col_nav_input = widgets.Text(
    description='Enter the column name for NAV values: ',
    placeholder='e.g., Net Asset Value',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

col_date_input = widgets.Text(
    description='Enter the column name for NAV date: ',
    placeholder='e.g., NAV date',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

n_years_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description='Rolling returns window (in years):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

run_button = widgets.Button(
    description='🚀 Run Analyzer',
    button_style='success'
)

In [ ]:
input_ready = False

In [ ]:
# @title Define button
def on_run_clicked(b):
    clear_output(wait=True)  # clear UI to focus on outputs
    display(fund_input, nav_path_input, n_years_slider,
            col_nav_input, col_date_input,
            run_button)

    # Capture inputs
    fund_name_temp = fund_input.value.strip()
    loc_nav_temp = nav_path_input.value.strip()
    n_years_temp = n_years_slider.value

    # Confirm
    print(f'✅ Starting analysis for: {fund_name_temp}')
    print(f'📂 Reading NAVs from: {loc_nav_temp}')
    print(f'📈 Rolling CAGR: {n_years_temp} years')

    input_ready = True

    #

run_button.on_click(on_run_clicked)

##Stage: Temp

In [ ]:
# @title
# Display UI
# display(fund_input, nav_path_input, n_years_slider,
#         col_nav_input, col_date_input,
#         run_button)

## Stage: Package Import

In [ ]:
# Core Data Libraries
import pandas as pd
import numpy as np

# Date handling
from datetime import timedelta
from bisect import bisect_right

# Visualization
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# File handling
import os
import glob


## Stage: Data Import

In [ ]:
# @title Capture inputs
#fund_name = fund_input.value.strip()
#loc_nav = nav_path_input.value.strip()
#n_years = n_years_slider.value

#col_nav = col_nav_input.value.strip()
#col_date = col_date_input.value.strip()

In [ ]:
# @title loc_nav error
#if not loc_nav:
 #   raise ValueError("Please enter a valid directory path for NAV files.")

In [ ]:
# @title
project_dir = '/content/project_mf_analyzer'
os.makedirs(project_dir, exist_ok=True)

print(f'📁 Project folder created at: {project_dir}')

In [ ]:
# # @title
# # Prompt user to enter path
# #loc_nav = str(input('Enter the directory where historical NAV data is present: ')).strip()

# # Check if path exists
# if not os.path.isdir(loc_nav):
#     raise FileNotFoundError(f'❌ The folder "{loc_nav}" does not exist. Please check the path and try again.')

# print(f'✅ NAV folder found at: {loc_nav}')


In [ ]:
# # @title Reading NAV files
# df_list = []

# for filename in os.listdir(loc_nav):
#     file_path = os.path.join(loc_nav, filename)

#     if filename.endswith(('.xls', '.xlsx', '.html')):
#         try:
#             # Try reading as HTML first for .xls and .html files
#             print(f'Attempting to read {filename} as HTML.')
#             dfs_html = pd.read_html(file_path)
#             if dfs_html:
#                 # Assuming the first table is the correct one
#                 df = dfs_html[0]
#                 # Flatten the MultiIndex columns
#                 df.columns = df.columns.get_level_values(-1)
#                 df.columns = df.columns.str.strip()
#             else:
#                 print(f'⚠️ Skipped HTML file: {filename} as no tables were found.')
#                 continue

#         except Exception as e_html:
#             # If reading as HTML fails, try reading as Excel
#             try:
#                 print(f'Attempting to read {filename} as Excel due to HTML read error: {e_html}')
#                 temp_df = pd.read_excel(file_path, header=None, engine='openpyxl')

#                 # Detect header row using user-defined col_date
#                 header_row_index = temp_df[temp_df.apply(
#                     lambda row: row.astype(str).str.contains(col_date, case=False).any(), axis=1
#                 )].index[0]

#                 df = pd.read_excel(file_path, skiprows=header_row_index, engine='openpyxl')

#             except Exception as e_excel:
#                 print(f'⚠️ Skipped file: {filename} due to error reading as both HTML and Excel: {e_excel}')
#                 continue


#     elif filename.endswith('.csv'):
#         try:
#             temp_df = pd.read_csv(file_path, header=None)

#             header_row_index = temp_df[temp_df.apply(
#                 lambda row: row.astype(str).str.contains(col_date, case=False).any(), axis=1
#             )].index[0]

#             df = pd.read_csv(file_path, skiprows=header_row_index)

#         except Exception as e:
#             print(f'⚠️ Skipped CSV file: {filename} due to error: {e}')
#             continue

#     else:
#         continue

#     df.columns = df.columns.astype(str).str.strip()

#     if col_date not in df.columns or col_nav not in df.columns:
#         print(f'⚠️ Skipped file (missing columns): {filename}')
#         continue

#     df = df[[col_date, col_nav]]
#     df = df.rename(columns={col_date: 'date', col_nav: 'nav'})

#     df['fund_name'] = fund_name
#     df['date'] = pd.to_datetime(df['date'], infer_datetime_format=True, errors='coerce')
#     df['nav'] = pd.to_numeric(df['nav'], errors='coerce')

#     df = df.dropna(subset=['fund_name', 'date', 'nav'])
#     df.drop_duplicates(inplace = True)

#     print ('After pre processing file: ', filename)
#     print ('Shape of file: ', df.shape)
#     print ('Distinct dates in file: ', df['date'].nunique())
#     print ('Range of dates in file: ', df['date'].min(), ' to ', df['date'].max())
#     print ('\n')

#     df_list.append(df)

# # Combine all NAV data
# print ('Combining all individual files.')
# if df_list:
#     df_nav_all = pd.concat(df_list, ignore_index=True).sort_values('date')
#     df_nav_all.drop_duplicates(inplace = True)
#     df_nav_all.sort_values('date', inplace=True, ascending = False)

#     print ('Shape of file: ', df_nav_all.shape)
#     print ('Distinct dates in file: ', df_nav_all['date'].nunique())
#     print ('Range of dates in file: ', df_nav_all['date'].min(), ' to ', df_nav_all['date'].max())

#     print(f'✅ Loaded NAV data from {len(df_list)} files')
#     display(df_nav_all.head())
# else:
#     print('❌ No dataframes were successfully loaded from the files.')

## Stage: Data Wrangling

In [ ]:
# 1. Make sure data is sorted chronologically
df_nav_all = df_nav_all.sort_values('date').reset_index(drop=True)

# 2. Ask user for number of years for rolling return
#n_years = int(input('Enter number of years (n) for rolling return analysis: ').strip())
print(f'✅ Preparing to calculate {n_years}-year rolling CAGR.')

# 3. Extract just the date and nav columns
dates = df_nav_all['date'].tolist()
navs = df_nav_all['nav'].tolist()

# 4. Define a utility function to find the nearest available NAV <= target date
def get_nearest_past_index(target_date, date_list):
    idx = bisect_right(date_list, target_date)
    return idx - 1 if idx > 0 else None


## Stage: Rolling Returns Analysis

In [ ]:
from datetime import timedelta

In [ ]:
# Copy for rolling result
rolling_cagr_list = []

# Loop through each date (starting from n_years onward)
for i, row in df_nav_all.iterrows():
    current_date = row['date']
    current_nav = row['nav']

    # Target date = n years back
    n_years_back = current_date - pd.DateOffset(years=n_years)

    # Find the nearest date <= n_years_back
    past_idx = get_nearest_past_index(n_years_back, dates)

    if past_idx is not None:
        past_nav = navs[past_idx]
        past_date = dates[past_idx]

        # Calculate CAGR
        cagr = (current_nav / past_nav) ** (1 / n_years) - 1

        rolling_cagr_list.append({
            'current_date': current_date,
            'past_date': past_date,
            f'cagr_{n_years}_years': cagr * 100,  # as %
            'current_nav' : current_nav,
            'past_nav': past_nav,
            'fund_name': row['fund_name']
        })

# Convert to DataFrame
df_rolling = pd.DataFrame(rolling_cagr_list)

# Final sort and preview
df_rolling = df_rolling.sort_values('current_date').reset_index(drop=True)

print(f'✅ Calculated {n_years}-year rolling CAGR for {len(df_rolling)} dates.')
df_rolling.head()


## Stage: Visualization & Export

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
col_cagr = f'cagr_{n_years}_years'

In [ ]:
p10 = df_rolling[col_cagr].quantile(0.10)
p50 = df_rolling[col_cagr].median()
p90 = df_rolling[col_cagr].quantile(0.90)

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipywidgets import Output

# --- Setup ---
output_plot = Output()

# Your existing plot functions
def plot_nav(df):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df['current_date'],
        y=df['current_nav'],
        mode='lines',
        name='NAV',
        line=dict(color='cyan')
    ))
    fig.update_layout(
        title='📈 NAV Over Time',
        xaxis_title='Date',
        yaxis_title='NAV',
        template='plotly_dark'
    )
    return fig



In [ ]:
#Rolling cagr plot
def plot_rolling_cagr(df, n_years):
    col_cagr = f'cagr_{n_years}_years'
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df['current_date'],
        y=df[col_cagr],
        mode='lines',
        name=f'{n_years}Y Rolling CAGR',
        line=dict(color='purple')
    ))

    # Percentile bands
    fig.add_hline(y=p10, line_dash='dot', line_color='red', annotation_text='10th percentile')
    fig.add_hline(y=p50, line_dash='dash', line_color='yellow', annotation_text='Median')
    fig.add_hline(y=p90, line_dash='dot', line_color='green', annotation_text='90th percentile')

    fig.update_layout(
        title=f'📉 Rolling {n_years}-Year CAGR',
        xaxis_title='Date',
        yaxis_title='CAGR (%)',
        template='plotly_dark'
    )
    return fig



In [ ]:
#Rolling cagr plot for multiple funds
def plot_rolling_cagr_mul_mf(df, n_years):
    if df['fund_name'].nunique() == 1:
      col_cagr = f'cagr_{n_years}_years'
      fig = go.Figure()
      fig.add_trace(go.Scatter(
          x=df['current_date'],
          y=df[col_cagr],
          mode='lines',
          name=f'{n_years}Y Rolling CAGR',
          line=dict(color='purple')
      ))

      # Percentile bands
      fig.add_hline(y=p10, line_dash='dot', line_color='red', annotation_text='10th percentile')
      fig.add_hline(y=p50, line_dash='dash', line_color='yellow', annotation_text='Median')
      fig.add_hline(y=p90, line_dash='dot', line_color='green', annotation_text='90th percentile')

      fig.update_layout(
          title=f'📉 Rolling {n_years}-Year CAGR',
          xaxis_title='Date',
          yaxis_title='CAGR (%)',
          template='plotly_dark'
      )
    else:
      col_cagr = f'cagr_{n_years}_years'
      fig = go.Figure()

      # Loop over each fund
      for fund in df['fund_name'].unique():
          fund_df = df[df['fund_name'] == fund]

          fig.add_trace(go.Scatter(
              x=fund_df['current_date'],
              y=fund_df[col_cagr],
              mode='lines',
              name=fund,
          ))

      fig.update_layout(
          title=f'📉 Rolling {n_years}-Year CAGR',
          xaxis_title='Date',
          yaxis_title='CAGR (%)',
          template='plotly_dark',
          legend=dict(
              orientation="h",        # horizontal legend
              yanchor="top",
              y=-0.25,                # move below the chart
              xanchor="center",
              x=0.5
          )
      )


    return fig



In [ ]:
# Compute percentiles
percentiles = df_rolling[col_cagr].quantile([0, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 1.0]).round(2) #.astype(int)

# Construct summary DataFrame
summary_df = pd.DataFrame({
    'fund_name': [df_rolling['fund_name'].iloc[0]],
    'start_date': [df_rolling['past_date'].min().date()],
    'end_date': [df_rolling['current_date'].max().date()],
    'metric': [f'{n_years}Y CAGR'],
    f'latest_{n_years}_year_cagr': [df_rolling[col_cagr].iloc[-1].round(2)],
    'mean': [round(df_rolling[col_cagr].mean(), 2)],
    'std_dev': [round(df_rolling[col_cagr].std(), 2)],
    'min': [percentiles.loc[0.00]],
    'p5': [percentiles.loc[0.05]],
    'p10': [percentiles.loc[0.10]],
    'p25': [percentiles.loc[0.25]],
    'median': [round(percentiles.loc[0.50], 2)],
    'p75': [percentiles.loc[0.75]],
    'p90': [percentiles.loc[0.90]],
    'p95': [percentiles.loc[0.95]],
    'max': [percentiles.loc[1.00]],
})

# Save both detailed and summary output
os.makedirs(f'{project_dir}/{fund_name}', exist_ok=True)
output_path = f'{project_dir}/{fund_name}/mf_analyzer_output.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_rolling.to_excel(writer, index=False, sheet_name='CAGR_Trail')
    summary_df.to_excel(writer, index=False, sheet_name='Summary')

print(f'✅ Excel saved to: {output_path}')
summary_df


#Stage: Building MF store

In [ ]:
!pip install llama-index --quiet
from llama_index.core import Document, VectorStoreIndex
import os

In [ ]:
os.listdir(project_dir)

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(project_dir)

def load_all_funds(base_dir=BASE_DIR):
    fund_store = {}

    for fund_folder in base_dir.iterdir():
        if fund_folder.is_dir():
            fund_name = fund_folder.name
            file_path = fund_folder / "mf_analyzer_output.xlsx"

            if file_path.exists():
                try:
                    fund_store[fund_name] = {
                        "rolling_nav_cagr": pd.read_excel(file_path, sheet_name="CAGR_Trail"),
                        "summary": pd.read_excel(file_path, sheet_name="Summary")
                    }
                except Exception as e:
                    print(f"Could not read {fund_name}: {e}")

    return fund_store

In [ ]:
fund_store = load_all_funds()

In [ ]:
fund_store.keys()

In [ ]:
# @title: Display Fund Summaries
for fund_name, data in fund_store.items():
    print(f"\nFund Name: {fund_name}")
    for key, value in data.items():
        print(f"  {key}:\n{value.head(2).to_string(index=False)}")

In [ ]:
def build_rag_context(fund_store):
    context_parts = []

    for fund_name, data in fund_store.items():
        rolling_nav_cagr = data["rolling_nav_cagr"]
        summary = data["summary"]

        context_parts.append(f"""
Fund Name: {fund_name}

Rolling NAV and CAGR Analysis:
{rolling_nav_cagr.to_string(index=False)}

Summary:
{summary.to_string(index=False)}
""")

    return "\n\n---\n\n".join(context_parts)

In [ ]:
fund_store_txt = build_rag_context(fund_store)

In [ ]:
print (fund_store_txt)

##Stage: Creating df for all fund's CAGR

In [ ]:
fund_store.keys()

In [ ]:
import pandas as pd

import re

def clean_fund_name(name):
    match = re.search(r"(.*?\bfund\b)", name, re.IGNORECASE)
    return match.group(1).strip() if match else name.strip()

all_data = []
all_data_summary = []

for k in fund_store.keys():
    temp = fund_store[k]['rolling_nav_cagr'].copy()
    temp['fund'] = temp['fund_name'].apply(lambda x: clean_fund_name(x))
    all_data.append(temp[["current_date", "fund_name", "fund", f"cagr_{n_years}_years"]])

    temp_2 = fund_store[k]['summary'].copy()
    temp_2['fund'] = temp_2['fund_name'].apply(lambda x: clean_fund_name(x))
    all_data_summary.append(temp_2)

fund_store_all_cagr = pd.concat(all_data, ignore_index=True)
fund_store_all_summary = pd.concat(all_data_summary, ignore_index=True)

In [ ]:
temp.columns

In [ ]:
fund_store_all_cagr.groupby('fund').count()

In [ ]:
fund_store_all_summary

# Stage: Performance Dashboard

In [ ]:
# @title Input for performance dashboard
graph_inp = input ('Enter "N" for NAV chart, "C" for rolling CAGR chart: ')

In [ ]:
# @title Performance chart
if graph_inp == 'N':
  plot_nav(df_rolling).show()
elif graph_inp == 'C':
  #plot_rolling_cagr(df_rolling, n_years).show()
  plot_rolling_cagr_mul_mf(fund_store_all_cagr, n_years).show()
else:
  print ('Invalid input')

In [ ]:
#@title #Summary
#summary_df
fund_store_all_summary.sort_values(by=f'mean', ascending=False)

#Stage: Distribution of returns

In [ ]:
#@title Returns Distribution Chart
import plotly.express as px

def plot_boxplot(df, n):

  fig = px.box(
      df,
      x="fund",
      y=f"cagr_{n}_years",
      color="fund",
      points="outliers",
      title=f"Rolling {n}Y CAGR Distribution Across Funds",
      template="plotly_dark"
  )

  fig.update_traces(
      boxmean=True,
      jitter=0.25,
      marker=dict(size=4, opacity=0.35)
  )

  fig.update_layout(
      height=550,
      width=900,
      title_x=0.5,
      showlegend=False,
      xaxis_title="",
      yaxis_title=f"cagr_{n}_years",
      font=dict(size=13),
      margin=dict(l=60, r=40, t=80, b=120)
  )

  fig.update_xaxes(
      tickangle=-25,
      showgrid=False
  )

  fig.update_yaxes(
      gridcolor="rgba(255,255,255,0.1)"
  )

  #Adding smart insights annotations
  stats = (
      df
      .groupby("fund")[f"cagr_{n}_years"]
      .agg(
          median="median",
          q1=lambda x: x.quantile(0.25),
          q3=lambda x: x.quantile(0.75),
          min_return="min",
          max_return="max"
      )
      .reset_index()
  )

  stats["spread"] = stats["q3"] - stats["q1"]

  best_median = stats.loc[stats["median"].idxmax()]
  most_consistent = stats.loc[stats["spread"].idxmin()]

  annotation_text = (
      f"<b>Smart Insight</b><br>"
      f"Highest median return: {best_median['fund']} "
      f"({best_median['median']:.2f}%)<br>"
      f"Most consistent fund: {most_consistent['fund']} "
      f"(IQR: {most_consistent['spread']:.2f}%)"
  )

  fig.add_annotation(
      text=annotation_text,
      xref="paper",
      yref="paper",
      x=0.99,
      y=0.98,
      showarrow=False,
      align="left",
      bgcolor="rgba(20,20,20,0.85)",
      bordercolor="rgba(255,255,255,0.25)",
      borderwidth=1,
      font=dict(size=12, color="white")
  )
  #fig.show()
  return fig

plot_boxplot(fund_store_all_cagr, n_years)

#Stage: Risk vs Returns Analysis

In [ ]:
#@title Risk - Returns Matrix
import plotly.express as px

def plot_risk_return_matrix(fund_store_all_summary, n_years):
    df = fund_store_all_summary.copy()

    x_col = f"std_dev"
    y_col = f"median"

    x_mid = df[x_col].median()
    y_mid = df[y_col].median()

    fig = px.scatter(
        df,
        x=x_col,
        y=y_col,
        text=None,
        hover_name="fund_name",
        title=f"Risk - Return Matrix ({n_years}Y Rolling CAGR)",
        template="plotly_dark"
    )

    fig.add_vline(x=x_mid, line_dash="dash", line_color="gray")
    fig.add_hline(y=y_mid, line_dash="dash", line_color="gray")

    fig.update_xaxes(range=[0, df[x_col].max() * 1.1])

    fig.update_yaxes(
        range=[min(0, df[y_col].min() * 1.1), df[y_col].max() * 1.1]
    )

    x_min = 0
    x_max = df[x_col].max() * 1.1

    y_min = 0
    y_max = df[y_col].max() * 1.1

    x_left = (x_min + x_mid) / 2
    x_right = (x_mid + x_max) / 2
    y_bottom = (y_min + y_mid) / 2
    y_top = (y_mid + y_max) / 2

    fig.add_annotation(x=x_left, y=y_top,
                      text="Low Risk<br>High Return<br><b>Best Zone</b>",
                      showarrow=False, font=dict(size=10, color="rgba(255,255,255,0.5)"))

    fig.add_annotation(x=x_right, y=y_top,
                      text="High Risk<br>High Return<br><b>Aggressive</b>",
                      showarrow=False, font=dict(size=10, color="rgba(255,255,255,0.5)"))

    fig.add_annotation(x=x_left, y=y_bottom,
                      text="Low Risk<br>Low Return<br><b>Stable</b>",
                      showarrow=False, font=dict(size=10, color="rgba(255,255,255,0.5)"))

    fig.add_annotation(x=x_right, y=y_bottom,
                      text="High Risk<br>Low Return<br><b>Weak Zone</b>",
                      showarrow=False, font=dict(size=10, color="rgba(255,255,255,0.5)"))

    fig.update_traces(
        marker=dict(
            size=8,
            opacity=0.85,
            line=dict(width=1, color="white")
        )
    )

    fig.update_xaxes(range=[x_min, x_max])
    fig.update_yaxes(range=[y_min, y_max])

    fig.update_traces(textposition="top center")

    fig.update_layout(
        xaxis_title="Risk: Volatility of Rolling CAGR",
        yaxis_title="Return: Median Rolling CAGR",
        showlegend=False
    )

    return fig

plot_risk_return_matrix(fund_store_all_summary, n_years)

#Stage: GenAI layer starts

In [ ]:
!pip install google-generativeai --quiet

In [ ]:
import google.generativeai as genai
import os




In [ ]:
from google.colab import userdata

In [ ]:
# Configure your API key
GOOGLE_API_KEY = userdata.get('google_api_key_myfirstproject')

In [ ]:
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
system_instruction = f"""
You are Scout, an AI assistant for mutual fund analysis.

Your role is to interpret mutual fund performance data in a clear, concise, intuitive, and investor-friendly way.

Mutual Fund Summary Data:
{fund_store_txt}

How to approach analysis:

- Treat the provided data as the primary source of insight.
- Focus on what the numbers are telling you about:
  - returns
  - consistency
  - volatility
  - downside risk
  - upside potential
  - overall performance

- Avoid being overly cautious or generic. Speak with clarity and conviction where the data supports it.

Single fund:
- Explain how the fund has performed over time.
- Highlight whether returns have been stable or volatile.
- Point out any meaningful downside risk.
- Summarize its overall profile.
- Give a simple rating (1–5) with a short justification.

Multiple funds:
- Compare them holistically, not just metric-by-metric.
- Identify which fund appears stronger and why.
- Call out tradeoffs (e.g., higher return vs higher risk).
- If the difference is small, say so clearly.

Style guidelines:
- Keep it natural, crisp, and easy to understand.
- Explain insights, not just numbers.
- Avoid sounding like a report — sound like a smart analyst explaining things.
- Highlight both strengths and weaknesses where relevant.

Additional context:
- If needed, you may use general knowledge about the fund (category, AMC, positioning) to enrich the explanation.
- However, let the provided performance data drive your conclusions.
- If there are multiple funds from different categories (like one from small cap, one from large cap), give a cautionary message that

Guardrails:
- If funds belong to different categories (e.g., large cap vs small cap), clearly warn that comparison is not apples-to-apples before proceeding.
- If metrics or time horizons differ across funds, do not compare them. Clearly explain why and suggest a fair comparison basis.
- Do not draw strong conclusions from limited or insufficient data.
- Do not rely on extreme values (max/min) alone — focus on typical performance (median, percentiles).
- Do not declare a fund better based only on higher returns — always consider risk, consistency, and downside.
- If differences between funds are small, state that clearly instead of forcing a winner.
- Do not overemphasize short-term performance over long-term trends unless explicitly asked.
- Do not assume or fill in missing data — call out gaps explicitly.
- Avoid giving investment advice (no “buy/sell”) — keep insights analytical and evidence-based.
- Where relevant, briefly indicate what type of investor a fund may suit (based on risk/return profile).

Tool Usage Rules:
- If the user asks for a chart, graph, visualization, or plot → you MUST call the appropriate tool
- If the user asks about risk vs return → you MUST call show_risk_return_matrix
- If the user asks about distribution, consistency, outliers, or spread → you MUST call show_boxplot
- If the user does not specify a time period → default to time period available in the data
- If the user does specify a time period and it is different from the time period available in the data, then return an error message mentioning that you do not have the data for that specified time period
- Do not write Python code.
- Do not show tool code.
- Do not say you cannot draw charts.
- After using a chart tool, briefly explain what the chart shows. Also interpret the graph for the user to understand it better.

End goal:
Help the user understand not just “which fund is better”, but *why*.
"""

In [ ]:
model = genai.GenerativeModel('gemini-2.5-flash', system_instruction = system_instruction)





In [ ]:
for m in genai.list_models():
    print(m.name, '->', m.supported_generation_methods)

#Stage: Creating AI agent with function calling capabilities

In [ ]:
!pip install -q langchain langchain-google-genai --quiet

In [ ]:
latest_fig = None

In [ ]:
#@title Defining tools for the AI agent
from langchain_core.tools import tool

@tool
def show_boxplot(n_years: int) -> dict:
    """
    Use this when the user asks for distribution, range, outliers,
    consistency, box plot, percentile spread, or rolling CAGR distribution.
    """

    fig = plot_boxplot(fund_store_all_cagr, n_years)

    global latest_fig
    latest_fig = fig

    return {
        "type": "plot",
        "figure": fig,
        "message": f"Here is the {n_years}Y rolling CAGR boxplot."
      }


@tool
def show_risk_return_matrix(n_years: int) -> dict:
    """
    Use this when the user asks for risk vs return, volatility vs return,
    risk-return matrix, low risk high return funds, or 2x2 quadrant analysis.
    """

    fig = plot_risk_return_matrix(fund_store_all_summary, n_years)

    global latest_fig
    latest_fig = fig

    return {
        "type": "plot",
        "figure": fig,
        "message": f"Here is the {n_years}Y risk-return matrix."
      }

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
#from langchain.agents import AgentExecutor
from langchain.agents import create_agent
#from langchain.agents.tool_calling_agent import create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

tools = [
    show_boxplot,
    show_risk_return_matrix
]

prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}")
])


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_instruction
)

In [ ]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Show me distribution of returns"}
    ]
})

#response

In [ ]:
display(latest_fig)

In [ ]:
from IPython.display import HTML

# Assuming latest_fig holds your Plotly figure
if latest_fig:
    html_output = latest_fig.to_html(include_plotlyjs='cdn', full_html=False)
    display(HTML(html_output))
else:
    print("No Plotly figure found in 'latest_fig'.")

In [ ]:
print (response['messages'][-1].content)

In [ ]:
display(latest_fig)

In [ ]:
# # Process all messages
# for msg in response["messages"]:

#     # Tool output
#     if hasattr(msg, "name"):

#         tool_output = msg.content
#         print (tool_output, '\n\n XXX')

#         if isinstance(tool_output, dict):

#             if tool_output.get("type") == "plot":

#                 display(tool_output["figure"])

#                 display(Markdown(
#                     f"**Scout:** {tool_output['message']}"
#                 ))

##Stage: Initiating Scout

In [ ]:
chat_session = model.start_chat(history=[])

##Stage: Designing widgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display

user_input = widgets.Textarea(
    placeholder="Hello! I am Scout - your personalised Mutual Fund AI Assistant. Ask me anything about the funds under consideration.",
    layout=widgets.Layout(width="90%", height="100px")
)

#display(user_input)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

common_questions = [
    "What mutual funds do you currently have data for?",
    "Explain the individual performance of each fund.",
    "Compare the funds holistically.",
    "Which fund has performed better overall?",
    "Which fund has better downside protection?",
    "Which fund is more consistent?",
    "During COVID, which fund dipped the least?",
    "Which fund has better risk-return tradeoff?",
    "If I am risk-averse, which fund should I prefer?",
    "What is the worst-case return observed for each fund under consideration?",
    "Which fund is more suitable for long-term investing and why?",
    "Which fund has delivered more stable returns over time?",
    "Is the higher return of any fund coming with significantly higher risk?",
    "Which fund shows more predictable performance across market cycles?",
    "Which fund has historically protected capital better during downturns?",
    "Are the differences between these funds meaningful or marginal?",
    "Which fund would suit a conservative investor vs an aggressive investor?",
    "Which fund has a better balance of risk and reward?",
    "Which fund would you choose if consistency is more important than returns?",
    "Which fund would you choose if maximizing returns is the goal?",
    "How do the worst-case rolling returns compare between the funds?",
    "Which fund shows higher volatility in rolling returns?",
    "Which fund has tighter return distribution (less variability)?",
    "When each fund hit its worst phase, how long did it take to recover? Include date ranges, key external triggers, and which fund handled it best.",
    "Give me a crisp verdict between these funds."
]

question_dropdown = widgets.Dropdown(
    options=["Select a suggested question..."] + common_questions,
    value="Select a suggested question...",
    description="Quick ask:",
    layout=widgets.Layout(width="90%")
)

In [ ]:
def on_question_selected(change):
    if change["new"] != "Select a suggested question...":
        user_input.value = change["new"]

question_dropdown.observe(on_question_selected, names="value")

In [ ]:
send_button = widgets.Button(
    description="Ask Scout",
    button_style="primary"
)

#display(send_button)

In [ ]:
#@title Define on_click function (OLD - using chat_session method)
# from IPython.display import display, Markdown

# def on_click(b):
#     user_query = user_input.value
#     #print("You:", user_query)
#     display(Markdown(f"**You:** {user_input.value}"))

#     if user_query.strip():  # Check if the input is not empty or just whitespace
#         response = chat_session.send_message(user_query)
#         display(Markdown(f"**Scout:** {response.text}"))
#         display(Markdown("<hr style='border:0.5px solid #444;'>"))
#     else:
#         display(Markdown("**Scout:** Please enter a question or command.\n"))

#     user_input.value = ""  # clear box


In [ ]:
# #@title Define on_click function (NEW - using agent.invoke method)
# from IPython.display import display, Markdown

# def on_click(b):
#     user_query = user_input.value
#     #print("You:", user_query)
#     display(Markdown(f"**You:** {user_input.value}"))

#     if user_query.strip():  # Check if the input is not empty or just whitespace
#         #response = chat_session.send_message(user_query)
#         response = agent.invoke({
#             "messages": [
#                 {"role": "user", "content": user_query}
#             ]
#         })
#         display(Markdown(f"**Scout:** {response['messages'][-1].content}"))
#         display(Markdown("<hr style='border:0.5px solid #444;'>"))
#     else:
#         display(Markdown("**Scout:** Please enter a question or command.\n"))

#     user_input.value = ""  # clear box


In [ ]:
#@title Define on_click function
from IPython.display import display, Markdown

# def extract_text(content):

#     # Plain string response
#     if isinstance(content, str):
#         return content

#     # Structured response
#     if isinstance(content, list):

#         texts = []

#         for item in content:

#             if isinstance(item, dict):

#                 if item.get("type") == "text":
#                     texts.append(item.get("text", ""))

#         return "\n".join(texts)

#     return str(content)

def extract_text(content):
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "\n".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict) and item.get("type") == "text"
        )

    return str(content)


def on_click(b):

    user_query = user_input.value.strip()

    display(Markdown(f"**You:** {user_query}"))

    if not user_query:
        display(Markdown("**Scout:** Please enter a question or command."))
        return

    response = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    })

    final_content = response["messages"][-1].content
    final_answer = extract_text(final_content)

    display(Markdown(f"**Scout:** {final_answer}"))
    display(Markdown("<hr style='border:0.5px solid #444;'>"))

    user_input.value = ""

#Ask Scout!

In [ ]:
#@title Let him cook

output_area = widgets.Output()

display(question_dropdown)
display(user_input)
display(send_button)
display(output_area)
send_button.on_click(on_click)

#Thank you!

In [ ]:
#@title Run this for visuals (if any created by Scout)
display(latest_fig)

##LLM as a judge

In [ ]:
# !pip install ragas datasets langchain-google-genai

In [ ]:
# import pandas as pd
# from datasets import Dataset

# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy

# from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# judge_llm = ChatGoogleGenerativeAI(
#     model="gemini-1.5-flash",
#     google_api_key=GOOGLE_API_KEY,
#     temperature=0
# )

In [ ]:
# test_questions = [
#     "Which fund looks better overall?",
#     "Which fund is more consistent?"
# ]

In [ ]:
# eval_rows = []

# for question in test_questions:
#     scout_prompt = f"""
# You are Scout, an AI assistant for mutual fund analysis.

# Mutual Fund Summary Data:
# {fund_store_txt}

# User question:
# {question}
# """
#     print('question')
#     scout_response = chat_session.send_message(scout_prompt)

#     eval_rows.append({
#         "question": question,
#         "answer": scout_response.text,
#         "contexts": [fund_store_txt]
#     })

# eval_df = pd.DataFrame(eval_rows)
# eval_df

In [ ]:
fund_store_txt

In [ ]:
fund_store

In [ ]:
# print(documents[0].text)
# print(documents[0].metadata)

In [ ]:
# fund_names_in_metadata = [doc.metadata['fund_name'] for doc in documents]
# print(fund_names_in_metadata)